# Interactive ADT Scene Scanpath 3D Viewer

This notebook mirrors the core functionality of `Experiments/visualization & Analysis/ADT/04_scene_scanpath_3d.py`, but renders the scene with Plotly so the 3D view can be rotated, panned, and zoomed interactively.

It supports selecting an ADT sequence, frame window, current frame, scanpath tail length, visible tracks, object boxes, skeleton, head/body trajectories, and ray-box hits.

In [6]:
from __future__ import annotations

import importlib.util
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import clear_output, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'Experiments').exists():
    REPO_ROOT = Path('/home/liumu/Github_Projects/adt_dataset_sandbox')

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

SCENE04_PATH = REPO_ROOT / 'Experiments' / 'visualization & Analysis' / 'ADT' / '04_scene_scanpath_3d.py'
spec = importlib.util.spec_from_file_location('scene_scanpath_04', SCENE04_PATH)
scene04 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = scene04
spec.loader.exec_module(scene04)

REPORTS_DIR = Path('/mnt/d/SparseGaze/ADT-Gaze-structured')
SPARSEGAZE_SEQUENCE_ROOT = Path('/home/liumu/Github_Projects/SparseGaze/outputs/eval/adt/sparsegaze/test/rollout/sequence_predictions')
HAGI_NPZ = Path('/home/liumu/Github_Projects/HAGI/save/head/hagi++_imputation/adt_low_framerate_sliding/sliding_primary_nsample20_framerate_6.npz')
HAGI_ADT_DATA = Path('/home/liumu/Github_Projects/HAGI/datasets/adt/gaze_head_adt.npy')

TRACK_LABELS = ['GT', 'SparseGaze', 'HAGI++']
TRACK_COLORS = scene04.TRACK_COLORS
BOX_EDGES = scene04.BOX_EDGES


In [7]:
def list_sequences(reports_dir: Path = REPORTS_DIR) -> list[str]:
    base = reports_dir / 'sequences'
    if not base.exists():
        return []
    return sorted(path.name for path in base.iterdir() if path.is_dir())


def sparsegaze_npz_for_sequence(sequence: str) -> Path:
    return SPARSEGAZE_SEQUENCE_ROOT / sequence / 'hz6_phase0.npz'


@lru_cache(maxsize=8)
def load_sequence_data(sequence: str, use_sparsegaze: bool = True, use_hagi: bool = True):
    context = scene04.load_scene_context(REPORTS_DIR, sequence)
    prediction_npz = sparsegaze_npz_for_sequence(sequence) if use_sparsegaze else None
    hagi_npz = HAGI_NPZ if use_hagi else None
    try:
        tracks, table = scene04.build_track_table(
            context=context,
            prediction_npz=prediction_npz,
            pred_key='pred_xyz',
            hagi_npz=hagi_npz,
            hagi_adt_data=HAGI_ADT_DATA,
        )
    except Exception as exc:
        print(f'warning: failed to load all prediction tracks for {sequence}: {exc}')
        tracks, table = scene04.build_track_table(
            context=context,
            prediction_npz=prediction_npz,
            pred_key='pred_xyz',
            hagi_npz=None,
            hagi_adt_data=HAGI_ADT_DATA,
        )
    return context, tracks, table


def sample_rows(rows: pd.DataFrame, stride: int) -> pd.DataFrame:
    stride = max(int(stride), 1)
    if len(rows) <= 1 or stride == 1:
        return rows
    sampled = rows.iloc[::stride].copy()
    if int(sampled['frame_index'].iloc[-1]) != int(rows['frame_index'].iloc[-1]):
        sampled = pd.concat([sampled, rows.tail(1)], ignore_index=True)
    return sampled


def finite_xyz(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    return scene04.finite_xyz(frame, columns)


def display_points(points: np.ndarray, vertical_axis: str, mirror_horizontal: bool) -> np.ndarray:
    return scene04.display_points(points, vertical_axis, mirror_horizontal)


In [8]:
def add_object_boxes(fig, object_rows, plotted, vertical_axis, mirror_horizontal, show_static=True, show_dynamic=True):
    if object_rows is None or object_rows.empty:
        return
    for rows, color, name, opacity, enabled in [
        (object_rows[object_rows['timestamp_ns'] == -1], '#6b7280', 'static object boxes', 0.22, show_static),
        (object_rows[object_rows['timestamp_ns'] != -1], '#2563eb', 'dynamic object boxes', 0.55, show_dynamic),
    ]:
        if not enabled or rows.empty:
            continue
        xs, ys, zs = [], [], []
        for _, row in rows.iterrows():
            corners = display_points(scene04.row_corners(row), vertical_axis, mirror_horizontal)
            for first, second in BOX_EDGES:
                xs.extend([corners[first, 0], corners[second, 0], None])
                ys.extend([corners[first, 1], corners[second, 1], None])
                zs.extend([corners[first, 2], corners[second, 2], None])
            plotted.append(corners)
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name=name, line=dict(color=color, width=2), opacity=opacity, hoverinfo='skip'))


def add_tracks(fig, rows, tracks, enabled_tracks, current_frame, vertical_axis, mirror_horizontal, plotted):
    for track in tracks:
        if track not in enabled_tracks:
            continue
        prefix = scene04.track_prefix(track)
        points = finite_xyz(rows, [f'{prefix}_x', f'{prefix}_y', f'{prefix}_z'])
        if len(points) == 0:
            continue
        points = display_points(points, vertical_axis, mirror_horizontal)
        color = TRACK_COLORS.get(track, "#778EF3")
        fig.add_trace(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='lines+markers', name=f'{track} scanpath',
            line=dict(color=color, width=7), marker=dict(size=4, color=color),
        ))
        current = rows[rows['frame_index'] == current_frame]
        current_points = finite_xyz(current, [f'{prefix}_x', f'{prefix}_y', f'{prefix}_z'])
        if len(current_points):
            current_points = display_points(current_points, vertical_axis, mirror_horizontal)
            fig.add_trace(go.Scatter3d(
                x=current_points[:, 0], y=current_points[:, 1], z=current_points[:, 2],
                mode='markers', name=f'{track} current',
                marker=dict(size=8, color=color, symbol='diamond'), showlegend=False,
            ))
        plotted.append(points)


def add_head_trajectory(fig, head, rows, current_frame, vertical_axis, mirror_horizontal, plotted):
    if head is None or head.empty:
        return
    timestamps = set(rows[rows['frame_index'] <= current_frame]['query_timestamp_ns'].astype(np.int64))
    head_rows = head[head['query_timestamp_ns'].isin(timestamps)]
    points = finite_xyz(head_rows, ['head_origin_scene_x_m', 'head_origin_scene_y_m', 'head_origin_scene_z_m'])
    if len(points) == 0:
        return
    points = display_points(points, vertical_axis, mirror_horizontal)
    fig.add_trace(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='lines', name='head/device trajectory', line=dict(color='#0f766e', width=5)))
    fig.add_trace(go.Scatter3d(x=points[-1:, 0], y=points[-1:, 1], z=points[-1:, 2], mode='markers', name='current head', marker=dict(color='#0f766e', size=5), showlegend=False))
    plotted.append(points)


def add_body_trajectory(fig, skeleton, rows, current_frame, vertical_axis, mirror_horizontal, plotted):
    if skeleton is None or skeleton.empty or 'frame_index' not in skeleton.columns:
        return
    frame_set = set(rows[rows['frame_index'] <= current_frame]['frame_index'].astype(int))
    skel_rows = skeleton[skeleton['frame_index'].astype(int).isin(frame_set)]
    points = finite_xyz(skel_rows, ['root_joint_scene_x_m', 'root_joint_scene_y_m', 'root_joint_scene_z_m'])
    if len(points) == 0:
        return
    points = display_points(points, vertical_axis, mirror_horizontal)
    fig.add_trace(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='lines', name='body/root trajectory', line=dict(color='#9333ea', width=4)))
    plotted.append(points)


def add_current_skeleton(fig, skeleton, skeleton_summary, current_frame, vertical_axis, mirror_horizontal, plotted):
    if skeleton is None or skeleton.empty or not skeleton_summary or 'frame_index' not in skeleton.columns:
        return
    nearest = skeleton.iloc[int(np.argmin(np.abs(skeleton['frame_index'].to_numpy(dtype=int) - current_frame)))]
    labels = skeleton_summary.get('joint_labels', [])
    connections = skeleton_summary.get('joint_connections', [])
    points = []
    for joint_index, label in enumerate(labels):
        safe = scene04.safe_joint_label(str(label))
        columns = [f'joint_{joint_index:02d}_{safe}_scene_x_m', f'joint_{joint_index:02d}_{safe}_scene_y_m', f'joint_{joint_index:02d}_{safe}_scene_z_m']
        if not all(column in nearest.index for column in columns):
            points.append([np.nan, np.nan, np.nan])
        else:
            points.append([float(nearest[column]) for column in columns])
    point_array = np.asarray(points, dtype=np.float64)
    finite = np.isfinite(point_array).all(axis=1)
    if not finite.any():
        return
    point_array = display_points(point_array, vertical_axis, mirror_horizontal)
    xs, ys, zs = [], [], []
    for first_joint, second_joint in connections:
        if first_joint >= len(point_array) or second_joint >= len(point_array):
            continue
        if not (finite[first_joint] and finite[second_joint]):
            continue
        xs.extend([point_array[first_joint, 0], point_array[second_joint, 0], None])
        ys.extend([point_array[first_joint, 1], point_array[second_joint, 1], None])
        zs.extend([point_array[first_joint, 2], point_array[second_joint, 2], None])
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name='current skeleton', line=dict(color='#7c3aed', width=4)))
    fig.add_trace(go.Scatter3d(x=point_array[finite, 0], y=point_array[finite, 1], z=point_array[finite, 2], mode='markers', name='skeleton joints', marker=dict(color='#a78bfa', size=3), showlegend=False))
    plotted.append(point_array[finite])


def add_current_hit(fig, rows, current_frame, vertical_axis, mirror_horizontal, plotted):
    current = rows[rows['frame_index'] == current_frame]
    if current.empty:
        return
    row = current.iloc[-1]
    if not bool(row.get('object_hit', False)):
        return
    point = np.asarray([row.get('hit_x_m'), row.get('hit_y_m'), row.get('hit_z_m')], dtype=float)
    if not np.isfinite(point).all():
        return
    point = display_points(point.reshape(1, 3), vertical_axis, mirror_horizontal)[0]
    label = row.get('hit_instance_name', 'current GT ray-box hit')
    fig.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode='markers', name=f'hit: {label}', marker=dict(color='#dc2626', size=8, symbol='diamond')))
    plotted.append(point.reshape(1, 3))


In [9]:
def build_interactive_scene_figure(
    sequence: str,
    start_frame: int,
    end_frame: int,
    current_frame: int,
    stride: int,
    tail_frames: int,
    enabled_tracks: tuple[str, ...],
    show_static_objects: bool,
    show_dynamic_objects: bool,
    show_head: bool,
    show_body: bool,
    show_skeleton: bool,
    show_hit: bool,
    max_static_objects: int,
    vertical_axis: str,
    mirror_horizontal: bool,
    view_elev: float,
    view_azim: float,
):
    context, tracks, table = load_sequence_data(sequence, 'SparseGaze' in enabled_tracks, 'HAGI++' in enabled_tracks)
    start_frame = max(0, int(start_frame))
    end_frame = min(int(end_frame), int(table['frame_index'].max()) + 1)
    if end_frame <= start_frame:
        end_frame = start_frame + 1
    window = scene04.select_window(table, start_frame, end_frame)
    current_frame = int(np.clip(current_frame, int(window['frame_index'].min()), int(window['frame_index'].max())))
    focus_rows = window[window['frame_index'] <= current_frame]
    tail = focus_rows[focus_rows['frame_index'] >= current_frame - int(tail_frames)].copy()
    tail = sample_rows(tail, stride)
    trajectory_rows = sample_rows(focus_rows, stride)

    focus_timestamp = int(focus_rows['query_timestamp_ns'].iloc[-1])
    object_rows = scene04.select_object_rows(
        context.objects,
        focus_timestamp_ns=focus_timestamp,
        category_filter='',
        exclude_category_filter='shelter',
        max_static_objects=int(max_static_objects),
    )

    fig = go.Figure()
    plotted = []
    add_object_boxes(fig, object_rows, plotted, vertical_axis, mirror_horizontal, show_static_objects, show_dynamic_objects)
    if show_head:
        add_head_trajectory(fig, context.head, trajectory_rows, current_frame, vertical_axis, mirror_horizontal, plotted)
    if show_body:
        add_body_trajectory(fig, context.skeleton, trajectory_rows, current_frame, vertical_axis, mirror_horizontal, plotted)
    if show_skeleton:
        add_current_skeleton(fig, context.skeleton, context.skeleton_summary, current_frame, vertical_axis, mirror_horizontal, plotted)
    active_tracks = [track for track in tracks if track in enabled_tracks]
    add_tracks(fig, tail, active_tracks, enabled_tracks, current_frame, vertical_axis, mirror_horizontal, plotted)
    if show_hit:
        add_current_hit(fig, window, current_frame, vertical_axis, mirror_horizontal, plotted)

    ranges = scene04.axis_ranges(plotted)
    x_label, y_label, z_label = scene04.display_axis_labels(vertical_axis, mirror_horizontal)
    fig.update_layout(
        title=f'{sequence} | frame={current_frame} | window=[{start_frame}, {end_frame}) | tail={tail_frames}',
        scene=dict(
            xaxis=dict(title=f'{x_label} [m]', range=ranges[0]),
            yaxis=dict(title=f'{y_label} [m]', range=ranges[1]),
            zaxis=dict(title=f'{z_label} [m]', range=ranges[2]),
            aspectmode='data',
            camera=scene04.plotly_camera(vertical_axis, view_elev, view_azim),
        ),
        height=760,
        legend=dict(x=0.01, y=0.99),
        margin=dict(l=0, r=0, t=42, b=0),
    )
    return fig


In [10]:
sequences = list_sequences()
if not sequences:
    raise RuntimeError(f'No ADT sequences found under {REPORTS_DIR / "sequences"}')

default_sequence = 'Apartment_release_decoration_skeleton_seq133_M1292'
if default_sequence not in sequences:
    default_sequence = sequences[0]

sequence_dropdown = widgets.Dropdown(options=sequences, value=default_sequence, description='sequence', layout=widgets.Layout(width='760px'))
start_input = widgets.BoundedIntText(value=149, min=0, max=100000, description='start', layout=widgets.Layout(width='180px'))
end_input = widgets.BoundedIntText(value=300, min=1, max=100000, description='end', layout=widgets.Layout(width='180px'))
current_input = widgets.BoundedIntText(value=299, min=0, max=100000, description='current', layout=widgets.Layout(width='180px'))
stride_input = widgets.BoundedIntText(value=1, min=1, max=100, description='stride', layout=widgets.Layout(width='160px'))
tail_input = widgets.BoundedIntText(value=30, min=0, max=5000, description='tail', layout=widgets.Layout(width='160px'))
max_objects_input = widgets.BoundedIntText(value=80, min=0, max=1000, description='objects', layout=widgets.Layout(width='180px'))
tracks_select = widgets.SelectMultiple(options=TRACK_LABELS, value=tuple(TRACK_LABELS), description='tracks', layout=widgets.Layout(width='260px', height='90px'))
show_static_checkbox = widgets.Checkbox(value=True, description='static boxes', indent=False)
show_dynamic_checkbox = widgets.Checkbox(value=True, description='dynamic boxes', indent=False)
show_head_checkbox = widgets.Checkbox(value=True, description='head traj', indent=False)
show_body_checkbox = widgets.Checkbox(value=True, description='body traj', indent=False)
show_skeleton_checkbox = widgets.Checkbox(value=True, description='skeleton', indent=False)
show_hit_checkbox = widgets.Checkbox(value=True, description='hit point', indent=False)
mirror_checkbox = widgets.Checkbox(value=True, description='mirror X', indent=False)
vertical_dropdown = widgets.Dropdown(options=['scene_y', 'scene_z', 'scene_x'], value='scene_y', description='up')
view_elev = widgets.FloatSlider(value=22.0, min=-90.0, max=90.0, step=1.0, description='elev', continuous_update=False)
view_azim = widgets.FloatSlider(value=-58.0, min=-180.0, max=180.0, step=1.0, description='azim', continuous_update=False)
render_button = widgets.Button(description='Render / update', button_style='primary', icon='refresh')
status = widgets.HTML()
out = widgets.Output()


def sync_frame_bounds(*_):
    try:
        _, _, table = load_sequence_data(sequence_dropdown.value)
        max_frame = int(table['frame_index'].max())
        start_input.max = max_frame
        end_input.max = max_frame + 1
        current_input.max = max_frame
        if end_input.value > max_frame + 1:
            end_input.value = max_frame + 1
        if current_input.value > max_frame:
            current_input.value = max_frame
        status.value = f'<span style="color:#555">loaded {sequence_dropdown.value}: {max_frame + 1} frames</span>'
    except Exception as exc:
        status.value = f'<span style="color:#b91c1c">failed to load sequence: {exc}</span>'


def render(_=None):
    with out:
        clear_output(wait=True)
        try:
            fig = build_interactive_scene_figure(
                sequence=sequence_dropdown.value,
                start_frame=start_input.value,
                end_frame=end_input.value,
                current_frame=current_input.value,
                stride=stride_input.value,
                tail_frames=tail_input.value,
                enabled_tracks=tuple(tracks_select.value),
                show_static_objects=show_static_checkbox.value,
                show_dynamic_objects=show_dynamic_checkbox.value,
                show_head=show_head_checkbox.value,
                show_body=show_body_checkbox.value,
                show_skeleton=show_skeleton_checkbox.value,
                show_hit=show_hit_checkbox.value,
                max_static_objects=max_objects_input.value,
                vertical_axis=vertical_dropdown.value,
                mirror_horizontal=mirror_checkbox.value,
                view_elev=view_elev.value,
                view_azim=view_azim.value,
            )
            display(fig)
            status.value = '<span style="color:#166534">rendered</span>'
        except Exception as exc:
            status.value = f'<span style="color:#b91c1c">render failed: {exc}</span>'
            raise

sequence_dropdown.observe(sync_frame_bounds, names='value')
render_button.on_click(render)
sync_frame_bounds()

controls = widgets.VBox([
    sequence_dropdown,
    widgets.HBox([start_input, end_input, current_input, stride_input, tail_input, max_objects_input]),
    widgets.HBox([tracks_select, widgets.VBox([show_static_checkbox, show_dynamic_checkbox, show_head_checkbox, show_body_checkbox, show_skeleton_checkbox, show_hit_checkbox])]),
    widgets.HBox([vertical_dropdown, mirror_checkbox, view_elev, view_azim, render_button]),
    status,
])

display(controls, out)
render()


Output()

Notes:

- The 3D plot is a normal Plotly scene: drag to rotate, scroll to zoom, right-drag to pan.
- SparseGaze and HAGI++ 3D points follow the same convention as script 04: predicted direction plus aligned GT depth.
- `stride` only downsamples displayed points/trajectories for interactivity; it does not change the underlying loaded data.
- Use the camera controls (`elev`, `azim`) only as an initial camera. After rendering, the Plotly view can be changed directly with the mouse.